# SSF2 RL — Exploration & Data Collection

This notebook connects to the instrumented SSF2 build, lets you poke at the
observation/action space, drive the character with scripted inputs, and record
trajectories you can later use for supervised learning (behavioral cloning) or
to sanity-check your own RL implementations.

**Prerequisites** (run once, from the repo root):
```bash
cd /Users/cachemiss/Documents/projects/reflash2-fork/reflash2
.venv/bin/pip install -e python          # makes ssf2_rl importable
```

**The game auto-launches.** `env.reset()` starts the game itself if it isn't
already running — no manual terminal step needed. Requires `AIR_SDK_HOME` set,
or an AIR SDK at `~/Developer/AIRSDK*`. Game output is logged to
`.macos/adl.log`; `ssf2_rl.stop_game()` quits a game started this way. To
start it manually instead:
```bash
AIR_SDK_HOME="$HOME/Developer/AIRSDK_51.3.3" bash tools/macos/run_macos.sh
```

**One env, one hierarchy.** Every slot is declared with a single class
(`Agent` / `Human` / `CPU` / bots), and everything runs through `env`:
`env.reset(players=..., stage=...)` → `env.run(frames)` for real-time
evaluation → `env.step()` to step through one frame at a time.

Select the repo's `.venv` as the notebook kernel (Cmd+Shift+P → "Notebook: Select Kernel" → `.venv`).

In [1]:
#   ZeroBot / FollowBot / ScriptedBot / PolicyBot — Python-driven bots.
# Every bot inherits a safe default of mask 0 (noop), so a taken-over slot
# never silently reverts to the in-game CPU.
# (PolicyBot is also available from ssf2_rl for neural-net policies.)
from ssf2_rl.bots import Agent, ZeroBot, FollowBot, ScriptedBot, ZeroBot
from ssf2_rl.players import CPU, Human, Character, Stage
from ssf2_rl import NOOP, LEFT, RIGHT, DOWN, SPECIAL   # controls bit constants

from ssf2_rl.env import SSF2Env


## 1. Connect & reset (programmatic match setup)

`reset()` restarts the match in-game and takes over the bot slots. Every slot
is declared with one class — the declaration IS the controller:

```python
env.reset(players={1: Agent("marth"), 2: CPU("samus", level=0)}, stage="battlefield")
```

- `Agent` — driven by `env.step(action)` (the RL path)
- `Human` — you play it in the game window
- `CPU` — the in-game AI at a level
- `ZeroBot` / `FollowBot` / `ScriptedBot` / `PolicyBot` — Python bots

`env.describe_matchup()` prints exactly who controls each slot. After reset,
use `env.run(frames)` to watch it play in real time, or loop `env.step()` to
step through one frame at a time.

In [2]:
env = SSF2Env()                      # agent_player=1 by default

# Programmatic match setup: declare every slot + the stage right in reset().
# Agent(...) = the RL slot, driven by env.step(action).
obs, info = env.reset(
    players={1: Agent(Character.Marth), 2: CPU(Character.ZeroSuitSamus, level=0)},
    stage=Stage.bf,
)
print(env.describe_matchup())
print("\nframe:", info["frame"], "| me:", info["me"]["name"], "| opp:", info["opp"]["name"])

# game starts, marth moves on his own, while samus plays like a cpu level 0. I would
# think that marth would stand still, but that isn't the case

[ssf2_rl] Launching SSF2 via /Users/cachemiss/Developer/AIRSDK_51.3.3/bin/adl (log: /Users/cachemiss/Documents/projects/reflash2-fork/reflash2/.macos/adl.log) ...
[ssf2_rl] Game is up; bridge listening on 127.0.0.1:4567.
stage: battlefield
P1: external agent (step-driven), character=marth
P2: in-game CPU level 0, character=zamus

frame: 2 | me: Marth | opp: Zero Suit Samus


In [ ]:
# human vs cpu

obs, info = env.reset(
    players={1: Human("marth"), 2: CPU("samus", level=9)},
)
print(env.describe_matchup())
env.run(frames=20*30)   # 20s (at 30fps)

# game restarts, p1 initially stands still (expected, supposed to be human control). I can control it (test passed).
# p2 plays like level 9 cpu (test passed).

stage: finaldestination
P1: human, character=marth
P2: in-game CPU level 9, character=samus
Playing for ~20s — take control of the game window!
run(): 600 frames, 0 dropped


{}

In [ ]:
# --- Human vs ZeroBot --------------------------------------------------------
# P1 is you; P2 is taken over by Python and held at mask 0 every frame —
# a perfectly still punching bag. Verify in the window: P2 never moves,
# and its `controls` stays 0 for every frame.

obs, info = env.reset(
    players={1: Human("marth"), 2: ZeroBot("samus")},
)
print(env.describe_matchup())
traj = env.run(frames=600, record=True)   # ~20s: play against the dummy

frames = traj[2]
ctrls = [next(c for c in r["state"]["chars"] if c["id"] == 2)["controls"] for r in frames]
print(f"P2 controls==0 on {sum(1 for c in ctrls if c == 0)}/{len(ctrls)} frames")

# p1 and p2 behave as expected for 20s. After that, p2 reverts to cpu (lvl 9?)

stage: finaldestination
P1: human, character=marth
P2: Python ZeroBot (zero), character=samus
run(): 600 frames, 0 dropped
P2 controls==0 on 600/600 frames


In [ ]:
# --- Scripted bot (dashdance) vs ZeroBot, 300 frames ------------------------
# P1 alternates left/right in short bursts (on_end="loop" keeps it going);
# P2 stands still. Watch P1 shuffle back and forth in the game window.
# Script entries are (controls mask, frames) — masks are bit constants from
# ssf2_rl.controls, combinable with | (e.g. DOWN | SPECIAL).

script = [
    (NOOP, 100),   # warm up past the entrance/countdown freeze
    (LEFT, 20),
    (RIGHT, 20),
    (LEFT, 20),
    (RIGHT, 20),
]

obs, info = env.reset(
    players={
        1: ScriptedBot(Character.Marth, script, on_end="loop"),
        2: ZeroBot(Character.Samus),
    },
)
print(env.describe_matchup())
traj = env.run(frames=150 + 300, record=True)

xs = [next(c for c in r["state"]["chars"] if c["id"] == 1)["x"] for r in traj[1]]

# works as intended, then reverts back to cpu (!, undesired))

stage: battlefield
P1: Python ScriptedBot (scripted), character=marth
P2: Python ZeroBot (zero), character=samus
run(): 450 frames, 0 dropped


In [ ]:
# --- CPU level 9 vs CPU level 9 ----------------------------------------------
# Pure spectating: both slots are the in-game AI. Python sends no inputs;
# run() only keeps the frame loop ticking in real time.
# (agent_player=2 here just picks which slot the obs/reward are relative to.)

obs, info = env.reset(
    players={1: CPU("marth", level=9), 2: CPU("samus", level=9)},
)
print(env.describe_matchup())
env.run(frames=600)   # watch ~20s of CPU vs CPU

# Works as intended

stage: finaldestination
P1: in-game CPU level 9, character=marth
P2: in-game CPU level 9, character=samus
run(): 600 frames, 0 dropped


{}

In [ ]:
# --- Human vs FollowBot (observation sanity check) --------------------------
# P1 is you; P2 walks toward you and stops inside a deadzone. If Samus
# visibly chases Marth around the stage, the bridge's x-position data (and
# therefore every observation built from it) is trustworthy.

obs, info = env.reset(
    players={1: Human("marth"), 2: FollowBot("samus", deadzone=30.0)},
)
print(env.describe_matchup())
traj = env.run(frames=600, record=True)   # ~20s: run around, get chased

# Works as intended, then p2 reverts back to cpu

stage: finaldestination
P1: human, character=marth
P2: Python FollowBot (follow), character=samus
run(): 600 frames, 0 dropped


NameError: name 'np' is not defined